# Transformer（GPT / ViT / MAE）

Transformer は、入力列の各要素が他の要素を参照して表現を更新するモデルです。文章なら単語やサブワード、画像ならパッチが列になります。鍵になる操作は self-attention です。各位置が query を出し、全位置の key と照合し、重要度に応じて value を混ぜます。

同じ attention でも、未来を隠すと GPT 型の次トークン予測になり、画像をパッチ列にすると ViT になり、パッチの一部を隠して復元すると MAE になります。違いは「入力をどう列にするか」と「どこを見せるか」にあります。

## ベクトル計算の土台

attention は内積、行列積、softmax で作れます。大きなライブラリを使わず、数値が追える関数から始めます。

In [ ]:
import math
import random


def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


def matvec(matrix, vector):
    return [dot(row, vector) for row in matrix]


def matmul(a, b):
    b_cols = list(zip(*b))
    return [[dot(row, col) for col in b_cols] for row in a]


def transpose(matrix):
    return [list(col) for col in zip(*matrix)]


def softmax(values):
    m = max(values)
    exps = [math.exp(v - m) for v in values]
    total = sum(exps)
    return [v / total for v in exps]


def weighted_sum(weights, vectors):
    return [sum(w * vec[i] for w, vec in zip(weights, vectors)) for i in range(len(vectors[0]))]


def layer_norm(vector, eps=1e-6):
    mean = sum(vector) / len(vector)
    var = sum((x - mean) ** 2 for x in vector) / len(vector)
    scale = math.sqrt(var + eps)
    return [(x - mean) / scale for x in vector]


def rounded(matrix, ndigits=3):
    return [[round(x, ndigits) for x in row] for row in matrix]


def print_matrix(name, matrix):
    print(name)
    for row in matrix:
        print(' ', [round(x, 3) for x in row])

## トークンをベクトルにする

Transformer は文字列を直接処理しません。語彙表で ID に変換し、ID ごとに埋め込みベクトルを持ちます。小さな文をベクトル列へ変換します。

In [ ]:
tokens = ['deep', 'models', 'learn', 'patterns']
embedding_table = {
    'deep': [1.0, 0.1, 0.0, 0.2],
    'models': [0.9, 0.2, 0.1, 0.0],
    'learn': [0.1, 1.0, 0.2, 0.2],
    'patterns': [0.0, 0.8, 0.3, 0.4],
}
X = [embedding_table[token] for token in tokens]
print_matrix('token embeddings', X)

## Q, K, V で参照先を選ぶ

各入力ベクトルから query, key, value を作ります。query と key の内積が大きいほど、その位置の value が強く混ざります。スケーリングは次元が大きいときに softmax が極端になりすぎるのを抑えます。

In [ ]:
Wq = [
    [0.8, 0.1, 0.0, 0.0],
    [0.0, 0.7, 0.2, 0.0],
    [0.1, 0.0, 0.6, 0.2],
]
Wk = [
    [0.7, 0.0, 0.1, 0.0],
    [0.1, 0.8, 0.0, 0.1],
    [0.0, 0.2, 0.7, 0.2],
]
Wv = [
    [0.6, 0.1, 0.0, 0.1],
    [0.0, 0.6, 0.2, 0.1],
    [0.1, 0.0, 0.5, 0.3],
]

def project(sequence, weights):
    return [matvec(weights, vector) for vector in sequence]


def attention(query, key, value, causal=False):
    scale = math.sqrt(len(query[0]))
    weights = []
    contexts = []
    for i, q in enumerate(query):
        scores = []
        for j, k in enumerate(key):
            score = dot(q, k) / scale
            if causal and j > i:
                score = -1e9
            scores.append(score)
        row = softmax(scores)
        weights.append(row)
        contexts.append(weighted_sum(row, value))
    return weights, contexts

Q = project(X, Wq)
K = project(X, Wk)
V = project(X, Wv)
weights, contexts = attention(Q, K, V)
print_matrix('attention weights', weights)
print_matrix('context vectors', contexts)

## 因果マスクで未来を隠す

GPT 型の decoder-only モデルは、位置 i の予測で i より右のトークンを見ません。マスクを入れると、左から右へ生成する条件付き確率を学べます。

In [ ]:
full_weights, _ = attention(Q, K, V, causal=False)
causal_weights, causal_contexts = attention(Q, K, V, causal=True)

print_matrix('full attention', full_weights)
print_matrix('causal attention', causal_weights)
print_matrix('causal context vectors', causal_contexts)

## 位置情報を足す

attention だけでは、同じベクトル集合の並び替えを区別しにくくなります。位置ベクトルを足すと、同じ単語でも出現位置に応じて表現が変わります。

In [ ]:
def sinusoidal_position(index, dim):
    values = []
    for k in range(dim):
        denominator = 10000 ** (2 * (k // 2) / dim)
        angle = index / denominator
        values.append(math.sin(angle) if k % 2 == 0 else math.cos(angle))
    return values


def add_position(sequence):
    return [[x + p for x, p in zip(vector, sinusoidal_position(i, len(vector)))] for i, vector in enumerate(sequence)]

order_a = [embedding_table[t] for t in ['deep', 'models']]
order_b = [embedding_table[t] for t in ['models', 'deep']]
print_matrix('order A without position', order_a)
print_matrix('order B without position', order_b)
print_matrix('order A with position', add_position(order_a))
print_matrix('order B with position', add_position(order_b))

## multi-head attention

複数の head は、違う射影で別々の関係を拾います。片方が語彙の近さ、片方が位置や役割の近さを拾うように重みを変えると、結合後の表現が豊かになります。

In [ ]:
head1 = {
    'q': Wq,
    'k': Wk,
    'v': Wv,
}
head2 = {
    'q': [
        [0.1, 0.6, 0.0, 0.2],
        [0.2, 0.1, 0.7, 0.0],
    ],
    'k': [
        [0.0, 0.7, 0.1, 0.0],
        [0.2, 0.0, 0.6, 0.2],
    ],
    'v': [
        [0.2, 0.5, 0.1, 0.0],
        [0.0, 0.1, 0.6, 0.3],
    ],
}

def run_head(sequence, params):
    q = project(sequence, params['q'])
    k = project(sequence, params['k'])
    v = project(sequence, params['v'])
    return attention(q, k, v, causal=True)

head1_weights, head1_context = run_head(X, head1)
head2_weights, head2_context = run_head(X, head2)
joined = [a + b for a, b in zip(head1_context, head2_context)]
print_matrix('head 1 weights', head1_weights)
print_matrix('head 2 weights', head2_weights)
print_matrix('concatenated head output', joined)

## 残差接続、正規化、feed-forward

Transformer block は attention の出力を足し戻し、正規化し、位置ごとの小さな MLP を通します。残差接続は情報の通り道を残し、正規化は値のスケールを安定させます。

In [ ]:
def relu(x):
    return max(0.0, x)


def feed_forward(vector):
    hidden_w = [
        [0.3, -0.2, 0.4, 0.1, 0.0],
        [0.1, 0.5, -0.1, 0.2, 0.3],
        [0.0, 0.2, 0.3, -0.3, 0.4],
        [0.2, 0.0, 0.1, 0.4, -0.2],
    ]
    output_w = [
        [0.4, 0.1, 0.0, 0.2],
        [0.0, 0.3, 0.2, 0.1],
        [0.2, -0.1, 0.4, 0.0],
        [0.1, 0.2, 0.0, 0.3],
        [0.0, 0.1, 0.3, 0.2],
    ]
    hidden = [relu(v) for v in matvec(transpose(hidden_w), vector)]
    return matvec(transpose(output_w), hidden)

block_input = X[2]
attended = causal_contexts[2] + [0.0]
residual_1 = [a + b for a, b in zip(block_input, attended)]
norm_1 = layer_norm(residual_1)
ff = feed_forward(norm_1)
residual_2 = [a + b for a, b in zip(norm_1, ff)]
norm_2 = layer_norm(residual_2)
print('block input      ', [round(x, 3) for x in block_input])
print('after attention  ', [round(x, 3) for x in norm_1])
print('after feedforward', [round(x, 3) for x in norm_2])

## ViT は画像をパッチ列にする

Vision Transformer は、画像を固定サイズのパッチへ分割し、各パッチをベクトルに射影します。以後は文章の token embedding と同じ形になります。

In [ ]:
image = [
    [0, 0, 4, 4],
    [0, 1, 4, 5],
    [7, 7, 2, 2],
    [8, 7, 2, 3],
]

def patchify(image, patch_size):
    patches = []
    for top in range(0, len(image), patch_size):
        for left in range(0, len(image[0]), patch_size):
            patch = []
            for r in range(top, top + patch_size):
                patch.extend(image[r][left:left + patch_size])
            patches.append(patch)
    return patches

patches = patchify(image, patch_size=2)
patch_projection = [
    [0.2, 0.0, 0.1, 0.0],
    [0.0, 0.2, 0.0, 0.1],
    [0.1, 0.0, 0.2, 0.0],
]
patch_tokens = [matvec(patch_projection, patch) for patch in patches]
print_matrix('image patches', patches)
print_matrix('patch tokens', patch_tokens)

## MAE は隠したパッチを復元する

Masked Autoencoder は、見えているパッチだけを encoder に入れ、隠したパッチを decoder で復元します。損失は隠した場所にだけかけます。

In [ ]:
def mae_mask(n, mask_ratio, seed):
    rng = random.Random(seed)
    indices = list(range(n))
    rng.shuffle(indices)
    n_mask = round(n * mask_ratio)
    masked = sorted(indices[:n_mask])
    visible = [i for i in range(n) if i not in masked]
    return visible, masked


def mean_patch(patches, visible_indices):
    dim = len(patches[0])
    return [sum(patches[i][j] for i in visible_indices) / len(visible_indices) for j in range(dim)]

visible, masked = mae_mask(len(patches), mask_ratio=0.5, seed=7)
prototype = mean_patch(patches, visible)
reconstructions = {i: prototype for i in masked}
loss_terms = []
for i in masked:
    target = patches[i]
    pred = reconstructions[i]
    loss_terms.append(sum((a - b) ** 2 for a, b in zip(target, pred)) / len(target))
mae_loss = sum(loss_terms) / len(loss_terms)
print('visible patch indices:', visible)
print('masked patch indices :', masked)
print('prototype prediction :', [round(x, 3) for x in prototype])
print('masked reconstruction loss:', round(mae_loss, 3))

## decoder-only の次トークン予測

GPT 型モデルは、左側の文脈から次のトークン分布を出します。下の小さな例では、因果 attention で作った文脈ベクトルを語彙ベクトルと照合し、次トークンの確率を出します。

In [ ]:
vocab = ['deep', 'models', 'learn', 'patterns', '<end>']
extended_embeddings = dict(embedding_table)
extended_embeddings['<end>'] = [0.0, 0.1, 0.9, 0.2]

def next_token_distribution(prefix):
    seq = [extended_embeddings[token] for token in prefix]
    seq = add_position(seq)
    q = project(seq, Wq)
    k = project(seq, Wk)
    v = project(seq, Wv)
    _, ctx = attention(q, k, v, causal=True)
    state = ctx[-1] + [0.0]
    scores = {token: dot(state, extended_embeddings[token]) for token in vocab}
    probs = softmax(list(scores.values()))
    return sorted(zip(vocab, probs), key=lambda item: item[1], reverse=True)

for prefix in [['deep'], ['deep', 'models'], ['deep', 'models', 'learn']]:
    dist = next_token_distribution(prefix)
    print('prefix:', ' '.join(prefix))
    for token, prob in dist[:3]:
        print(f'  {token:8s} {prob:.3f}')

## 実務で性能を左右する設計点

Transformer の性能は、attention の式だけで決まりません。tokenizer、位置表現、mask、head 数、層数、学習データ、損失関数、推論時の制約が組み合わさってモデルの性質を決めます。

文章生成なら因果マスクと次トークン予測が中核になります。画像分類ならパッチ化と class token、自己教師あり表現学習なら mask と再構成対象が重要になります。GPT、ViT、MAE は、共通部品に対して入力形式、mask、目的関数を変えた設計として整理できます。